# CrystalFormer-CSP：固定化学式与空间群生成晶体

选择 **Python (highk_bec)** 内核，按顺序运行。直接调用本地原版 `main.py` 和真实预训练权重；Windows 使用 JAX CPU。

在第一个代码单元修改化学式、空间群和采样数。每次运行创建新目录，原始采样、日志、组成筛选结果、独立 CIF 和检查表保存在一起。

本 Notebook 的条件是**化学式和空间群**。文字结构描述、配位和连接方式没有进入模型；输出为未弛豫候选。

In [ ]:
from ast import literal_eval
from datetime import datetime
from pathlib import Path
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Composition, Structure
from pymatgen.io.cif import CifWriter
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

FORMULA = "LiCu2OF3"
SPACE_GROUP = 14
NUM_SAMPLES = 8
BATCH_SIZE = 8
SEED = 42
TEMPERATURE = 1.0
TOP_P = 1.0

PROJECT_DIR = Path("vendor/CrystalFormer-CSP").resolve()
CHECKPOINT = PROJECT_DIR / "checkpoints/alex20s/csp/epoch_046000.pkl"
OUTPUT_DIR = PROJECT_DIR / f"outputs/notebook_{FORMULA}_{datetime.now():%Y%m%d_%H%M%S_%f}"


## 1. 调用原版模型采样

首次运行包含 JAX 编译，CPU 通常需要约 1–3 分钟。详细输出保存在 `sampling.log`。`--save_path` 末尾的 `/` 用于兼容原版路径处理。

使用当前 Notebook 内核的 `sys.executable` 调用模型。权重不存在时会停止，避免原版脚本退回随机初始化。

In [ ]:
assert CHECKPOINT.is_file(), f"未找到预训练权重：{CHECKPOINT}"
OUTPUT_DIR.mkdir(parents=True)
process_env = {**os.environ, "JAX_PLATFORM_NAME": "cpu", "PYTHONIOENCODING": "utf-8"}

print(f"正在生成 {FORMULA}，空间群 {SPACE_GROUP}，样本数 {NUM_SAMPLES}……", flush=True)
with (OUTPUT_DIR / "sampling.log").open("w", encoding="utf-8") as log:
    subprocess.run(
        [sys.executable, "-u", "main.py", "--optimizer", "none",
         "--restore_path", str(CHECKPOINT), "--formula", FORMULA,
         "--spacegroup", str(SPACE_GROUP), "--num_samples", str(NUM_SAMPLES),
         "--batchsize", str(BATCH_SIZE), "--num_io_process", "1",
         "--seed", str(SEED), "--temperature", str(TEMPERATURE), "--top_p", str(TOP_P),
         "--save_path", OUTPUT_DIR.as_posix() + "/"],
        cwd=PROJECT_DIR, env=process_env, stdout=log, stderr=subprocess.STDOUT, check=True,
    )
print(f"采样完成：{OUTPUT_DIR}")


## 2. 用官方脚本展开对称操作并筛选组成

原版模型不保证每个样本的元素配比正确。此步骤调用官方转换脚本，按目标约分组成筛选。其 `_struct.csv` 文件的 `cif` 列实际保存 pymatgen Structure 字典，下一步再导出独立 CIF。

In [ ]:
with (OUTPUT_DIR / "conversion.log").open("w", encoding="utf-8") as log:
    subprocess.run(
        [sys.executable, "-u", "scripts/awl2struct.py",
         "--output_path", OUTPUT_DIR.as_posix() + "/",
         "--formula", FORMULA, "--num_io_process", "1"],
        cwd=PROJECT_DIR, env=process_env, stdout=log, stderr=subprocess.STDOUT, check=True,
    )

raw_samples = pd.read_csv(OUTPUT_DIR / f"output_{FORMULA}.csv")
structure_rows = pd.read_csv(OUTPUT_DIR / f"output_{FORMULA}_struct.csv")
print(f"原始采样 {len(raw_samples)} 个；组成匹配 {len(structure_rows)} 个。")


## 3. 导出 CIF，回读并检查

保留所有组成匹配候选，逐个记录实际空间群和最短不同原子间周期距离，不把这些检查称为“结构正确”或“稳定”。空间群使用 `symprec=0.01 Å`、`angle_tolerance=5°`。

CIF 导出关闭晶胞 refinement；随后重新读取文件执行检查。这里不执行能量弛豫或完整配位验证。

In [ ]:
target_composition = Composition(FORMULA).reduced_composition
records = []
for index, value in enumerate(structure_rows["cif"], 1):
    structure = Structure.from_dict(literal_eval(value))
    cif_path = OUTPUT_DIR / f"candidate_{index:03d}.cif"
    CifWriter(structure, symprec=0.01, refine_struct=False).write_file(cif_path)
    structure = Structure.from_file(cif_path)
    analyzer = SpacegroupAnalyzer(structure, symprec=0.01, angle_tolerance=5)
    distances = structure.distance_matrix
    np.fill_diagonal(distances, np.inf)
    records.append({
        "file": cif_path.name,
        "formula": structure.composition.reduced_formula,
        "atoms": len(structure),
        "space_group": analyzer.get_space_group_number(),
        "space_group_symbol": analyzer.get_space_group_symbol(),
        "min_distance_A": round(float(distances.min()), 4),
        "formula_matches": structure.composition.reduced_composition == target_composition,
        "space_group_matches": analyzer.get_space_group_number() == SPACE_GROUP,
    })

checks = pd.DataFrame(records, columns=[
    "file", "formula", "atoms", "space_group", "space_group_symbol",
    "min_distance_A", "formula_matches", "space_group_matches",
])
checks.to_csv(OUTPUT_DIR / "checks.csv", index=False, encoding="utf-8-sig")
print(f"已导出 {len(checks)} 个 CIF；实际空间群匹配 {int(checks['space_group_matches'].sum())} 个。")
display(checks.head(20))


## 使用说明

- 示例为 `LiCu2OF3`、空间群 `14`。也可改成 `Ba2ZrO4`、空间群 `139`。
- `NUM_SAMPLES` 是原始采样数；组成筛选后的 CIF 数量可能更少，甚至为零。
- 输出目录在采样单元末尾显示。`candidate_*.cif` 为候选文件，`checks.csv` 为检查表；原始模型输出也保留。
- 若调用失败，查看该次目录里的 `sampling.log` 或 `conversion.log`。
- 修改参数后，从第一个代码单元重新顺序运行。完整重跑会创建新目录。
- 如需按化学式自动选择空间群，在采样命令中将 `"--spacegroup", str(SPACE_GROUP)` 改为 `"--K", "10"`。此时检查表中的 `space_group_matches` 只是与所填 `SPACE_GROUP` 的比较，不再表示模型被要求固定该空间群。

这些文件仍需按具体任务进行短接触筛选、配位检查及能量弛豫；本 Notebook 不改动模型算法或自动修复结构。